In [117]:
import networkx as nx
import numpy as np
import re
import math
import pandas as pd
import matplotlib.pyplot as plt
import random
from collections import Counter
from vem_sbm_new_version import VEM_SBM as VEM_SBM_NEW
from tqdm.auto import tqdm
from ILouvain_Gower import ML2Gower
from sklearn.metrics.cluster import adjusted_rand_score as ARI
from sa_cluster import sa_cluster
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from stoc import stoc
import pickle
col_dict = {0: 'red', 1: 'blue', 2: 'green', 3: 'orange', 4:'purple', 5:'black', 6:'yellow', 7:'pink', 8:'lightblue',9:'lime', 10:'sienna',11:'indigo'}

In [134]:
def VEM_SBM_best(A, Xg=None, Xc=None, K=2, n_restarts=20, base_seed=0, max_iter=200, tol = 1e-6,verbose=False):
    best = None
    best_seed = None
    for s in range(base_seed, base_seed + n_restarts):
        np.random.seed(s)
        fit = VEM_SBM_NEW(A, Xg=Xg, Xc=Xc, K=K, max_iter=200, tol = 1e-6, verbose=False)
        if best is None or fit['ELBO'][-1] > best['ELBO'][-1]:
            best = fit
            best_seed = s
    best['seed'] = best_seed
    return best

# First scenario

In [135]:
fits = {}
ilouvain_aris = []
sa_aris = []
stoc_aris = []

for sed in tqdm(range(500)):
    random.seed(sed)
    np.random.seed(sed)
    size_comm = [50,50,50]
    probs = [[0.6, 0.2, 0.05],[0.2,0.5,0.1],[0.05,0.1,0.5]]
    G=nx.stochastic_block_model(sizes=size_comm,p=probs,directed=False)

    Xg=np.zeros((150,2))
    Xg[:50,] = np.random.multivariate_normal(mean=[0.5, 0.5], cov=[[0.09, 0.001], [0.001, 0.09]], size=50)
    Xg[50:100,] = np.random.multivariate_normal(mean=[2.5, 2.5], cov=[[0.08, 0.002], [0.002, 0.09]], size=50)
    Xg[100:,] = np.random.multivariate_normal(mean=[-1.5, 2.5], cov=[[0.09, 0.003], [0.003, 0.08]], size=50)
    Xg=pd.DataFrame(Xg)
    A=nx.adjacency_matrix(G).todense()
    
    labels = []
    for n,i in enumerate(size_comm):
        labels += [n]*i

    Xc = pd.DataFrame(index=range(Xg.shape[0]), columns=['attr_cat1','attr_cat2'])
    Xc.loc[:49,'attr_cat1'] = random.choices([1,2,3], k=50, weights=[0.8,0.15,0.05])
    Xc.loc[50:99,'attr_cat1'] = random.choices([1,2,3], k=50, weights=[0.15,0.8,0.05])
    Xc.loc[100:,'attr_cat1'] = random.choices([1,2,3], k=50, weights=[0.05,0.15,0.8])

    Xc.loc[:49,'attr_cat2'] = random.choices([1,2,3,4], k=50, weights=[0.7,0.15,0.05,0.1])
    Xc.loc[50:99,'attr_cat2'] = random.choices([1,2,3,4], k=50, weights=[0.15,0.7,0.05,0.1])
    Xc.loc[100:,'attr_cat2'] = random.choices([1,2,3,4], k=50, weights=[0.05,0.15,0.7,0.1])

    fit = VEM_SBM_best(A, Xg=Xg, Xc=Xc, K=3, max_iter=50, tol=1e-8, verbose=False, n_restarts=10)
    fits[sed] = fit

    # ILOUVAIN
    attributes = {}
    for node in G.nodes():
        attributes[node] = {
            0: float(Xg.iloc[node, 0]),     # continuous
            1: float(Xg.iloc[node, 1]),     # continuous
            2: str(Xc.iloc[node, 0]),       # categorical
            3: str(Xc.iloc[node, 1]),       # categorical
        }

    attr_types = {0: 'continuous', 1: 'continuous',
                2: 'categorical', 3: 'categorical'}

    ml = ML2Gower(G, attributes, {n: n for n in G.nodes()}, attr_types)
    partition_ilouv = ml.findPartition()
    ilouvain_aris.append(ARI(list(partition_ilouv.values()), labels))

    # SA
    result_sa = sa_cluster(G, k=3, Xg=Xg, Xc=Xc, n_bins=10, 
                        l=20, c=0.15, sigma=0.5, verbose=False)
    sa_aris.append(adjusted_rand_score(labels, result_sa['labels']))

    # STOC
    result_stoc = stoc(G, Xg=Xg, Xc=Xc, tau=0.4, l=2)

    stoc_aris.append(ARI(labels, result_stoc['labels']))


with open('fits_scenario1_10restarts.pkl', 'wb') as f:
    pickle.dump(fits, f)

with open('ilouv_aris_scenario1_10restarts.pkl', 'wb') as f:
    pickle.dump(ilouvain_aris, f)

with open('sa_aris_scenario1_10restarts.pkl', 'wb') as f:
    pickle.dump(sa_aris, f)

with open('stoc_aris_scenario1_10restarts.pkl', 'wb') as f:
    pickle.dump(stoc_aris, f)

  0%|          | 0/500 [00:00<?, ?it/s]

/home/domenicocangemi/Desktop/Data Lab/Paper nostri/SBM con attributi/vem_sbm_new_version.py:176: RuntimeWarning: invalid value encountered in scalar multiply
  term_g += r[i, k] * log_mvnorm(Xg[i, :], mu_list[k], Sigma_list[k])
/home/domenicocangemi/Desktop/Data Lab/Paper nostri/SBM con attributi/vem_sbm_new_version.py:86: RuntimeWarning: invalid value encountered in matmul
  qf = float((diff.T @ Sinv @ diff)[0, 0])


In [136]:
size_comm = [50,50,50]

labels = []
for n,i in enumerate(size_comm):
    labels += [n]*i

theta_true = [[0.6, 0.2, 0.05],[0.2,0.5,0.1],[0.05,0.1,0.5]]
means_true = [[0.5, 0.5],[2.5,2.5],[-1.5,2.5]]
sigmas_true = [[[0.09, 0.001], [0.001, 0.09]], [[0.08, 0.002], [0.002, 0.09]], [[0.09, 0.003], [0.003, 0.08]]]
phi1_true = [[0.8,0.15,0.05],[0.15,0.8,0.05],[0.05,0.15,0.8]]
phi2_true =  [[0.7,0.15,0.05,0.1],[0.15,0.7,0.05,0.1],[0.05,0.15,0.7,0.1]]

In [137]:
from scipy.optimize import linear_sum_assignment

def match_labels(true_labels, pred_labels, K):
    cost = np.zeros((K, K))
    for k in range(K):
        for l in range(K):
            cost[k, l] = -np.sum((true_labels == k) & (pred_labels == l))
    row_ind, col_ind = linear_sum_assignment(cost)
    return col_ind

def mse(true, est):
    return np.mean((np.array(true) - np.array(est))**2)

In [ ]:
# with open('fits_scenario1_10restarts.pkl', 'rb') as f:
#    fits = pickle.load(f)

K = 3
results = {
    'ari': [],
    'theta': [],
    **{f'mu_{k}': [] for k in range(K)},
    **{f'sigma_{k}': [] for k in range(K)},
    'phi1': [],
    'phi2': [],
}

for sed in range(500):
    fit = fits[sed]
    pred = np.argmax(fit['r'], axis=1)
    perm = match_labels(np.array(labels), pred, K)
    
    Theta_aligned = fit['Theta'][np.ix_(perm, perm)]
    mu_aligned = [fit['mu_list'][perm[k]] for k in range(K)]
    Sigma_aligned = [fit['Sigma_list'][perm[k]] for k in range(K)]
    phi_aligned = [phi[perm, :] for phi in fit['phi_list']]
    
    results['ari'].append(ARI(labels, pred))
    results['theta'].append(mse(theta_true, Theta_aligned))
    results['phi1'].append(mse(phi1_true, phi_aligned[0]))
    results['phi2'].append(mse(phi2_true, phi_aligned[1]))
    
    for k in range(K):
        results[f'mu_{k}'].append(mse(means_true[k], mu_aligned[k]))
        results[f'sigma_{k}'].append(mse(sigmas_true[k], Sigma_aligned[k]))

In [139]:
summary = pd.DataFrame({key: {'mean': np.mean(vals), 'std': np.std(vals)} 
                         for key, vals in results.items()}).T
print(summary.round(6))

             mean       std
ari      1.000000  0.000000
theta    0.000095  0.000060
mu_0     0.001757  0.001833
mu_1     0.001729  0.001674
mu_2     0.001737  0.001726
sigma_0  0.000228  0.000185
sigma_1  0.000209  0.000169
sigma_2  0.000212  0.000180
phi1     0.002294  0.001535
phi2     0.002389  0.001455


# Second scenario

In [141]:
fits = {}
ilouvain_aris = []
sa_aris = []
stoc_aris = []

for sed in tqdm(range(500)):
    random.seed(sed)
    np.random.seed(sed)

    size_comm = [50,50,50]
    probs = [[0.4, 0.2, 0.1],[0.2,0.4,0.2],[0.1,0.2,0.4]]
    G=nx.stochastic_block_model(sizes=size_comm,p=probs,directed=False)

    Xg=np.zeros((150,2))
    Xg[:50,] = np.random.multivariate_normal(mean=[0.5, 0.5], cov=[[0.09, 0.001], [0.001, 0.09]], size=50)
    Xg[50:100,] = np.random.multivariate_normal(mean=[2.5, 2.5], cov=[[0.08, 0.002], [0.002, 0.09]], size=50)
    Xg[100:,] = np.random.multivariate_normal(mean=[-1.5, 2.5], cov=[[0.09, 0.003], [0.003, 0.08]], size=50)
    Xg=pd.DataFrame(Xg)
    A=nx.adjacency_matrix(G).todense()

    labels = []
    for n,i in enumerate(size_comm):
        labels += [n]*i

    Xc = pd.DataFrame(index=range(Xg.shape[0]), columns=['attr_cat1','attr_cat2'])
    Xc.loc[:49,'attr_cat1'] = random.choices([1,2,3], k=50, weights=[0.8,0.15,0.05])
    Xc.loc[50:99,'attr_cat1'] = random.choices([1,2,3], k=50, weights=[0.15,0.8,0.05])
    Xc.loc[100:,'attr_cat1'] = random.choices([1,2,3], k=50, weights=[0.05,0.15,0.8])

    Xc.loc[:49,'attr_cat2'] = random.choices([1,2,3,4], k=50, weights=[0.7,0.15,0.05,0.1])
    Xc.loc[50:99,'attr_cat2'] = random.choices([1,2,3,4], k=50, weights=[0.15,0.7,0.05,0.1])
    Xc.loc[100:,'attr_cat2'] = random.choices([1,2,3,4], k=50, weights=[0.05,0.15,0.7,0.1])

    fit = VEM_SBM_best(A, Xg=Xg, Xc=Xc, K=3, max_iter=50, tol=1e-8, verbose=False, n_restarts=10)
    fits[sed] = fit

    # ILOUVAIN
    attributes = {}
    for node in G.nodes():
        attributes[node] = {
            0: float(Xg.iloc[node, 0]),     # continuous
            1: float(Xg.iloc[node, 1]),     # continuous
            2: str(Xc.iloc[node, 0]),       # categorical
            3: str(Xc.iloc[node, 1]),       # categorical
        }

    attr_types = {0: 'continuous', 1: 'continuous',
                2: 'categorical', 3: 'categorical'}

    ml = ML2Gower(G, attributes, {n: n for n in G.nodes()}, attr_types)
    partition_ilouv = ml.findPartition()
    ilouvain_aris.append(ARI(list(partition_ilouv.values()), labels))

    # SA
    result_sa = sa_cluster(G, k=3, Xg=Xg, Xc=Xc, n_bins=10, 
                        l=20, c=0.15, sigma=0.5, verbose=False)
    sa_aris.append(adjusted_rand_score(labels, result_sa['labels']))

    # STOC
    result_stoc = stoc(G, Xg=Xg, Xc=Xc, tau=0.4, l=2)

    stoc_aris.append(ARI(labels, result_stoc['labels']))


with open('fits_scenario2_10restarts.pkl', 'wb') as f:
    pickle.dump(fits, f)

with open('ilouv_aris_scenario2.pkl', 'wb') as f:
    pickle.dump(ilouvain_aris, f)

with open('sa_aris_scenario2.pkl', 'wb') as f:
    pickle.dump(sa_aris, f)

with open('stoc_aris_scenario2.pkl', 'wb') as f:
    pickle.dump(stoc_aris, f)

  0%|          | 0/500 [00:00<?, ?it/s]

In [142]:
size_comm = [50,50,50]

labels = []
for n,i in enumerate(size_comm):
    labels += [n]*i

theta_true = [[0.4, 0.2, 0.1],[0.2,0.4,0.2],[0.1,0.2,0.4]]
means_true = [[0.5, 0.5],[2.5,2.5],[-1.5,2.5]]
sigmas_true = [[[0.09, 0.001], [0.001, 0.09]], [[0.08, 0.002], [0.002, 0.09]], [[0.09, 0.003], [0.003, 0.08]]]
phi1_true = [[0.8,0.15,0.05],[0.15,0.8,0.05],[0.05,0.15,0.8]]
phi2_true =  [[0.7,0.15,0.05,0.1],[0.15,0.7,0.05,0.1],[0.05,0.15,0.7,0.1]]

In [143]:
# with open('fits_scenario2_10restarts.pkl', 'rb') as f:
#    fits = pickle.load(f)

K = 3
results = {
    'ari': [],
    'theta': [],
    **{f'mu_{k}': [] for k in range(K)},
    **{f'sigma_{k}': [] for k in range(K)},
    'phi1': [],
    'phi2': [],
}

for sed in range(500):
    fit = fits[sed]
    pred = np.argmax(fit['r'], axis=1)
    perm = match_labels(np.array(labels), pred, K)
    
    Theta_aligned = fit['Theta'][np.ix_(perm, perm)]
    mu_aligned = [fit['mu_list'][perm[k]] for k in range(K)]
    Sigma_aligned = [fit['Sigma_list'][perm[k]] for k in range(K)]
    phi_aligned = [phi[perm, :] for phi in fit['phi_list']]
    
    results['ari'].append(ARI(labels, pred))
    results['theta'].append(mse(theta_true, Theta_aligned))
    results['phi1'].append(mse(phi1_true, phi_aligned[0]))
    results['phi2'].append(mse(phi2_true, phi_aligned[1]))
    
    for k in range(K):
        results[f'mu_{k}'].append(mse(means_true[k], mu_aligned[k]))
        results[f'sigma_{k}'].append(mse(sigmas_true[k], Sigma_aligned[k]))

In [145]:
summary = pd.DataFrame({key: {'mean': np.mean(vals), 'std': np.std(vals)} 
                         for key, vals in results.items()}).T
print(summary.round(6))

             mean       std
ari      1.000000  0.000000
theta    0.000102  0.000061
mu_0     0.001757  0.001833
mu_1     0.001729  0.001674
mu_2     0.001737  0.001726
sigma_0  0.000228  0.000185
sigma_1  0.000209  0.000169
sigma_2  0.000212  0.000180
phi1     0.002240  0.001439
phi2     0.002451  0.001266


# Third scenario

In [147]:
fits = {}
ilouvain_aris = []
sa_aris = []
stoc_aris = []

for sed in tqdm(range(500)):
    random.seed(sed)
    np.random.seed(sed)

    size_comm = [50,50,50]
    probs = [[0.6, 0.2, 0.05],[0.2,0.5,0.1],[0.05,0.1,0.5]]
    G=nx.stochastic_block_model(sizes=size_comm,p=probs,directed=False)

    Xg=np.zeros((150,2))
    Xg[:50,] = np.random.multivariate_normal(mean=[0, 0.5], cov=[[0.09, 0.003], [0.003, 0.08]], size=50)
    Xg[50:100,] = np.random.multivariate_normal(mean=[0.5, 0.5], cov=[[0.08, 0.005], [0.005, 0.07]], size=50)
    Xg[100:,] = np.random.multivariate_normal(mean=[1, 1], cov=[[0.07, 0.001], [0.001, 0.085]], size=50)
    Xg=pd.DataFrame(Xg)
    A=nx.adjacency_matrix(G).todense()

    labels = []
    for n,i in enumerate(size_comm):
        labels += [n]*i

    Xc = pd.DataFrame(index=range(Xg.shape[0]), columns=['attr_cat1','attr_cat2'])
    Xc.loc[:49,'attr_cat1'] = random.choices([1,2,3], k=50, weights=[0.33,0.33,0.34])
    Xc.loc[50:99,'attr_cat1'] = random.choices([1,2,3], k=50, weights=[0.35,0.5,0.15])
    Xc.loc[100:,'attr_cat1'] = random.choices([1,2,3], k=50, weights=[0.35,0.15,0.5])

    Xc.loc[:49,'attr_cat2'] = random.choices([1,2,3,4], k=50, weights=[0.25,0.25,0.25,0.25])
    Xc.loc[50:99,'attr_cat2'] = random.choices([1,2,3,4], k=50, weights=[0.25,0.6,0.05,0.1])
    Xc.loc[100:,'attr_cat2'] = random.choices([1,2,3,4], k=50, weights=[0.05,0.7,0.15,0.1])

    fit = VEM_SBM_best(A, Xg=Xg, Xc=Xc, K=3, max_iter=50, tol=1e-8, verbose=False, n_restarts=10)
    fits[sed] = fit

    # ILOUVAIN
    attributes = {}
    for node in G.nodes():
        attributes[node] = {
            0: float(Xg.iloc[node, 0]),     # continuous
            1: float(Xg.iloc[node, 1]),     # continuous
            2: str(Xc.iloc[node, 0]),       # categorical
            3: str(Xc.iloc[node, 1]),       # categorical
        }

    attr_types = {0: 'continuous', 1: 'continuous',
                2: 'categorical', 3: 'categorical'}

    ml = ML2Gower(G, attributes, {n: n for n in G.nodes()}, attr_types)
    partition_ilouv = ml.findPartition()
    ilouvain_aris.append(ARI(list(partition_ilouv.values()), labels))

    # SA
    result_sa = sa_cluster(G, k=3, Xg=Xg, Xc=Xc, n_bins=10, 
                        l=20, c=0.15, sigma=0.5, verbose=False)
    sa_aris.append(adjusted_rand_score(labels, result_sa['labels']))

    # STOC
    result_stoc = stoc(G, Xg=Xg, Xc=Xc, tau=0.4, l=2)

    stoc_aris.append(ARI(labels, result_stoc['labels']))


with open('fits_scenario3_10restarts.pkl', 'wb') as f:
    pickle.dump(fits, f)

# with open('ilouv_aris_scenario3.pkl', 'wb') as f:
#     pickle.dump(ilouvain_aris, f)

# with open('sa_aris_scenario3.pkl', 'wb') as f:
#     pickle.dump(sa_aris, f)

# with open('stoc_aris_scenario3.pkl', 'wb') as f:
#     pickle.dump(stoc_aris, f)

  0%|          | 0/500 [00:00<?, ?it/s]

In [148]:
with open('ilouv_aris_scenario3.pkl', 'wb') as f:
    pickle.dump(ilouvain_aris, f)

with open('sa_aris_scenario3.pkl', 'wb') as f:
    pickle.dump(sa_aris, f)

with open('stoc_aris_scenario3.pkl', 'wb') as f:
    pickle.dump(stoc_aris, f)

In [149]:
size_comm = [50,50,50]

labels = []
for n,i in enumerate(size_comm):
    labels += [n]*i

theta_true = [[0.6, 0.2, 0.05],[0.2,0.5,0.1],[0.05,0.1,0.5]]
means_true = [[0, 0.5],[0.5,0.5],[1,1]]
sigmas_true = [[[0.09, 0.003], [0.003, 0.08]], [[0.08, 0.005], [0.005, 0.07]], [[0.07, 0.001], [0.001, 0.085]]]
phi1_true = [[0.33,0.33,0.34],[0.35,0.5,0.15],[0.35,0.15,0.5]]
phi2_true =  [[0.25,0.25,0.25,0.25],[0.25,0.6,0.05,0.1],[0.05,0.7,0.15,0.1]]

In [150]:
# with open('fits_scenario3_10restarts.pkl', 'rb') as f:
#    fits = pickle.load(f)

K = 3
results = {
    'ari': [],
    'theta': [],
    **{f'mu_{k}': [] for k in range(K)},
    **{f'sigma_{k}': [] for k in range(K)},
    'phi1': [],
    'phi2': [],
}

for sed in range(500):
    fit = fits[sed]
    pred = np.argmax(fit['r'], axis=1)
    perm = match_labels(np.array(labels), pred, K)
    
    Theta_aligned = fit['Theta'][np.ix_(perm, perm)]
    mu_aligned = [fit['mu_list'][perm[k]] for k in range(K)]
    Sigma_aligned = [fit['Sigma_list'][perm[k]] for k in range(K)]
    phi_aligned = [phi[perm, :] for phi in fit['phi_list']]
    
    results['ari'].append(ARI(labels, pred))
    results['theta'].append(mse(theta_true, Theta_aligned))
    results['phi1'].append(mse(phi1_true, phi_aligned[0]))
    results['phi2'].append(mse(phi2_true, phi_aligned[1]))
    
    for k in range(K):
        results[f'mu_{k}'].append(mse(means_true[k], mu_aligned[k]))
        results[f'sigma_{k}'].append(mse(sigmas_true[k], Sigma_aligned[k]))

In [151]:
summary = pd.DataFrame({key: {'mean': np.mean(vals), 'std': np.std(vals)} 
                         for key, vals in results.items()}).T
print(summary.round(6))

             mean       std
ari      0.999920  0.001267
theta    0.000095  0.000060
mu_0     0.001654  0.001714
mu_1     0.001533  0.001490
mu_2     0.001582  0.001574
sigma_0  0.000203  0.000163
sigma_1  0.000163  0.000134
sigma_2  0.000176  0.000150
phi1     0.004379  0.002633
phi2     0.002946  0.001520
